In [6]:
from ultralytics import YOLO

subset_yaml_path="/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det_20percent.yaml"

print("\n⚔️ [Exp 5.1] Baseline (ImageNet) on 20% Data...")
model_base = YOLO('yolo11n.pt')
model_base.train(
    data=subset_yaml_path,
    epochs=40,
    batch=16,
    imgsz=640,
    project='result_exp1',
    name='5.1_FewShot_Baseline',
    device='0',
    exist_ok=True,
    val=True
)


⚔️ [Exp 5.1] Baseline (ImageNet) on 20% Data...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det_20percent.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=5.1_FewShot_Baseline, nbs=64, nms=False, opset=Non

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f2afcedfe80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [5]:
import os
import random
import glob
import yaml
from ultralytics import YOLO

# ================= 配置路径 =================
BASE_DIR = "/root/autodl-tmp/exp1/NEU_DET_YOLO"
IMG_DIR = os.path.join(BASE_DIR, "images/train") # 假设这是你的图片目录
TRAIN_TXT_PATH = os.path.join(BASE_DIR, "train.txt")

# ================= 1. 修复：自动生成 train.txt =================
print("🚀 正在扫描所有训练图片...")
# 扫描所有 jpg 图片
all_imgs = glob.glob(os.path.join(IMG_DIR, "*.jpg")) + glob.glob(os.path.join(IMG_DIR, "*.png"))

if len(all_imgs) == 0:
    print(f"❌ 错误：在 {IMG_DIR} 下没有找到图片！请检查路径。")
else:
    print(f"✅ 找到 {len(all_imgs)} 张图片，正在生成全量列表 train.txt...")
    # 写入 train.txt
    with open(TRAIN_TXT_PATH, 'w') as f:
        f.write('\n'.join(all_imgs))

# ================= 2. 制作 20% 小样本数据集 =================
print("🚀 正在构建 20% 小样本数据集 (Few-Shot)...")

# 随机抽取 20%
subset_size = int(len(all_imgs) * 0.2)
subset_imgs = random.sample(all_imgs, subset_size)

print(f"📊 全量数据: {len(all_imgs)} -> 抽样 20%: {len(subset_imgs)}")

# 保存新的 20% txt
subset_txt_path = os.path.join(BASE_DIR, "train_20percent.txt")
with open(subset_txt_path, 'w') as f:
    f.write('\n'.join(subset_imgs))

# 生成新的 yaml (保持验证集不变)
# 注意：这里 names 列表需要和你原始 neu_det.yaml 保持一致
subset_yaml_content = f"""
path: {BASE_DIR}
train: {subset_txt_path}  # 指向 20% 数据列表
val: images/val         # 指向验证集文件夹 (或者 val.txt 如果你有的话)
names:
  0: crazing
  1: inclusion
  2: patches
  3: pitted_surface
  4: rolled-in_scale
  5: scratches
"""
subset_yaml_path = os.path.join(BASE_DIR, "neu_det_20percent.yaml")
with open(subset_yaml_path, 'w') as f:
    f.write(subset_yaml_content)

print(f"✅ 小样本配置已生成: {subset_yaml_path}")

# ================= 3. 实验 A: Baseline (ImageNet) on 20% =================
print("\n⚔️ [Exp 5.1] Baseline (ImageNet) on 20% Data...")
model_base = YOLO('yolo11n.pt')
model_base.train(
    data=subset_yaml_path,
    epochs=40,
    batch=16,
    imgsz=640,
    project='result_exp1',
    name='5.1_FewShot_Baseline',
    device='0',
    exist_ok=True,
    val=True
)

# ================= 4. 实验 B: Defect Spectrum Transfer on 20% =================
# 加载 Exp 2.3 的权重
PRETRAINED_WEIGHTS = "/root/autodl-tmp/exp1/result_exp1/2.3_DS_Pretrain/weights/best.pt"

print("\n🛡️ [Exp 5.2] Defect Spectrum Transfer on 20% Data...")
if os.path.exists(PRETRAINED_WEIGHTS):
    model_ds = YOLO(PRETRAINED_WEIGHTS)
    model_ds.train(
        data=subset_yaml_path,
        epochs=40,
        batch=16,
        imgsz=640,
        project='result_exp1',
        name='5.2_FewShot_DS_Transfer',
        device='0',
        freeze=10, # 冻结骨干
        lr0=0.005,
        exist_ok=True,
        val=True
    )
    print("\n🏆 小样本对比实验结束！请对比 5.1 和 5.2 的 mAP。")
else:
    print(f"❌ 警告：未找到 Exp 2.3 的权重文件: {PRETRAINED_WEIGHTS}，请先运行 Exp 2.3。")

🚀 正在扫描所有训练图片...
✅ 找到 1440 张图片，正在生成全量列表 train.txt...
🚀 正在构建 20% 小样本数据集 (Few-Shot)...
📊 全量数据: 1440 -> 抽样 20%: 288
✅ 小样本配置已生成: /root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det_20percent.yaml

⚔️ [Exp 5.1] Baseline (ImageNet) on 20% Data...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det_20percent.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01

In [1]:
from ultralytics import YOLO
import os

# ================= 配置区域 =================
# 1. 指向 NEU-DET (我们要回目标数据集了)
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det_20percent.yaml"

# 2. 关键！加载 Exp 2 训练好的权重
# 请确保这个路径是存在的
PRETRAINED_WEIGHTS = "/root/autodl-tmp/exp1/result_exp1/2_Severstal_Pretrain/weights/best.pt"

# ================= 开始训练 =================
print(f"🚀 开始 Exp 5.3: Clean Transfer (迁移学习)...")
print(f"Loading weights from: {PRETRAINED_WEIGHTS}")

# 加载预训练权重
model = YOLO(PRETRAINED_WEIGHTS) 

# 训练参数
model.train(
    data=NEU_YAML,
    epochs=50,             # 50 轮微调
    batch=16,              
    imgsz=640,
    workers=4,
    project='result_exp1',
    name='5.3_Fewshot_S_Transfer', # 实验名
    device='0',
    exist_ok=True,
    val=True               # ✅ 这里要把验证打开，我们要看最终分数！
)

print("\n🏆 Exp 5.3 迁移学习完成！")

🚀 开始 Exp 5.3: Clean Transfer (迁移学习)...
Loading weights from: /root/autodl-tmp/exp1/result_exp1/2_Severstal_Pretrain/weights/best.pt
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det_20percent.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_ex

In [2]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. SA-LDP 引擎 (小样本适配版) =================
class SA_LDP_Engine:
    def __init__(self, model, epsilon=50.0, delta=1e-5):
        self.model = model
        self.epsilon = epsilon
        self.delta = delta
        self.layer_roles = self._identify_layers()

    def _identify_layers(self):
        roles = {}
        for name, _ in self.model.named_parameters():
            if any(f"model.{i}." in name for i in range(10)): roles[name] = "backbone"
            elif "Detect" in name or "head" in name: roles[name] = "head"
            else: roles[name] = "neck"
        return roles

    def step(self, current_epoch):
        if current_epoch < 5: return # Warmup

        current_device = next(self.model.parameters()).device
        sensitivities = []
        param_groups = []
        names_list = []

        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                sensitivities.append(p.grad.norm(2).item())
                param_groups.append(p)
                names_list.append(name)
        
        if not sensitivities: return

        # Backbone 0.5, Head 2.0
        factors = []
        for n in names_list:
            role = self.layer_roles.get(n, "neck")
            if role == "backbone": factors.append(0.5) 
            elif role == "head":   factors.append(2.0) 
            else:                  factors.append(1.0)
            
        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            # 小样本下梯度比较脆弱，裁剪阈值给宽松一点
            clip_val = 5.0 
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            c = np.sqrt(2 * np.log(1.25 / self.delta))
            sigma = c * clip_val / (layer_eps + 1e-8)
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1

# ================= 2. 训练器 =================
class PrivacyTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = SA_LDP_Engine(model, epsilon=50.0) 
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 6 =================
SUBSET_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det_20percent.yaml"
# 加载你刚才跑出来的 Defect Spectrum 权重 (Exp 5.2 的结果，不是 2.3 的)
# 我们用 Exp 5.2 (在20%数据上微调过的) 作为起点，或者用 2.3 (纯预训练)
# 推荐：用 2.3 (纯预训练) 作为起点，重新进行隐私微调
DS_PRETRAIN = "/root/autodl-tmp/exp1/result_exp1/2.3_DS_Pretrain/weights/best.pt"

print("🚀 开始 Exp 6: 隐私保护下的小样本大对决...")

# --- Group A: Baseline + Privacy ---
print("\n⚔️ [Exp 6.1] Baseline (ImageNet) + Privacy...")
trainer_base = PrivacyTrainer(overrides={
    'model': 'yolo11n.pt',
    'data': SUBSET_YAML,
    'epochs': 30, # 小样本+隐私，不用跑太久
    'batch': 16,
    'imgsz': 640,
    'project': 'Thesis_Exp',
    'name': '6.1_Privacy_Baseline',
    'device': '0',
    'exist_ok': True
})
trainer_base.train()

# --- Group B: Transfer + Privacy ---
print("\n🛡️ [Exp 6.2] Transfer (Defect Spectrum) + Privacy...")
if os.path.exists(DS_PRETRAIN):
    trainer_ds = PrivacyTrainer(overrides={
        'model': DS_PRETRAIN,
        'data': SUBSET_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'Thesis_Exp',
        'name': '6.2_Privacy_Transfer',
        'device': '0',
        'exist_ok': True,
        'freeze': 10 # 保护好 Scratches 的特征！
    })
    trainer_ds.train()
else:
    print("❌ 找不到预训练权重")

🚀 开始 Exp 6: 隐私保护下的小样本大对决...

⚔️ [Exp 6.1] Baseline (ImageNet) + Privacy...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det_20percent.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=6.1_Privacy_Baseline, nb